# Condense Bible JSON formatting
Converts pretty-printed (sparse) `{ "hebrew": ..., "greek": [...], "latvian": [...] }` entries
to single-line-per-word compact format. Overwrites in place. Skips files already in condensed format.

In [ ]:
import json
from pathlib import Path

BIBLE_ROOT = "bible"

def is_sparse(text: str) -> bool:
    """True if any hebrew_word entry spans multiple lines (pretty-printed)."""
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if '"hebrew"' in line and '"greek"' not in line:
            return True
    return False

def condense(data: list) -> str:
    """Render the JSON with each hebrew_word entry on one line."""
    lines = ["["]
    for vi, verse in enumerate(data):
        lines.append("  {")
        lines.append('    "hebrew_words": [')
        words = verse.get("hebrew_words", [])
        for wi, w in enumerate(words):
            comma = "," if wi < len(words) - 1 else ""
            entry = json.dumps(w, ensure_ascii=False)
            lines.append(f"      {entry}{comma}")
        lines.append("    ],")
        lo_g = json.dumps(verse.get("leftover_greek", []), ensure_ascii=False)
        lo_l = json.dumps(verse.get("leftover_latvian", []), ensure_ascii=False)
        lines.append(f'    "leftover_greek": {lo_g},')
        lines.append(f'    "leftover_latvian": {lo_l}')
        verse_comma = "," if vi < len(data) - 1 else ""
        lines.append(f"  }}{verse_comma}")
    lines.append("]")
    return "\n".join(lines) + "\n"

json_files = sorted(Path(BIBLE_ROOT).glob("**/*.json"))
converted = skipped = errors = 0

for path in json_files:
    text = path.read_text(encoding="utf-8")
    if not is_sparse(text):
        skipped += 1
        continue
    try:
        data = json.loads(text)
        if not isinstance(data, list):
            print(f"  [skip] {path} — not a list")
            skipped += 1
            continue
        path.write_text(condense(data), encoding="utf-8")
        print(f"  [ok]   {path}")
        converted += 1
    except Exception as e:
        print(f"  [err]  {path} — {e}")
        errors += 1

print(f"\nDone. Converted={converted}  Already condensed={skipped}  Errors={errors}")